In [ ]:
import json

In [2]:
import pandas as pd

In [ ]:
def get_column_names(schemas, table_name, sorting_key='column_position'):  # function to get column names for a given dataset sorted by a specified key
    column_details = schemas.get(table_name) # or schemas[table_name] # get schema for the specified dataset
    columns = sorted(column_details, key=lambda col: col[sorting_key]) # sort columns by the specified key
    return list(map(lambda col: col['column_name'], columns)) # or [col['column_name'] for col in columns] # extract and return column names as a list

In [ ]:
schemas = json.load(open('C:\\Users\\subha\\Documents\\GitHub\\data_engineering_essentials_udemy\\retail_db\\schemas.json')) # load schema definitions from a JSON file

In [ ]:
orders_columns = get_column_names(schemas, 'orders')  # getting column names for orders dataset

In [ ]:
orders_df = pd.read_csv('retail_db/orders/part-00000', names=orders_columns) # Load orders data

In [ ]:
customers_columns = get_column_names(schemas, 'customers') # Get customers column names

In [ ]:
customers_df = pd.read_csv('retail_db/customers/part-00000', names=customers_columns) # Load customers data

In [ ]:
orders_df.head() # Display first few rows of orders dataframe


,order_id,order_date,order_customer_id,order_status
0,1,2013-07-25 00:00:00.0,11599,CLOSED
1,2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
2,3,2013-07-25 00:00:00.0,12111,COMPLETE
3,4,2013-07-25 00:00:00.0,8827,CLOSED
4,5,2013-07-25 00:00:00.0,11318,COMPLETE


In [ ]:
customers_df.head() # Display first few rows of customers dataframe

,customer_id,customer_fname,customer_lname,customer_email,customer_password,customer_street,customer_city,customer_state,customer_zipcode
0,1,Richard,Hernandez,XXXXXXXXX,XXXXXXXXX,6303 Heather Plaza,Brownsville,TX,78521
1,2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126
2,3,Ann,Smith,XXXXXXXXX,XXXXXXXXX,3422 Blue Pioneer Bend,Caguas,PR,725
3,4,Mary,Jones,XXXXXXXXX,XXXXXXXXX,8324 Little Common,San Marcos,CA,92069
4,5,Robert,Hudson,XXXXXXXXX,XXXXXXXXX,10 Crystal River Mall,Caguas,PR,725


In [17]:
orders_df = orders_df.set_index('order_customer_id') # Set index to order_customer_id for joining
customers_df = customers_df.set_index('customer_id') # Set index to customer_id for joining

In [47]:
customers_orders_df = customers_df.\
    join(orders_df, how='inner') # Join customers with orders on customer_id and order_customer_id

In [48]:
customers_orders_df # Display the resulting joined dataframe

,customer_fname,customer_lname,customer_email,customer_password,customer_street,customer_city,customer_state,customer_zipcode,order_id,order_date,order_status
customer_id,,,,,,,,,,,
1,Richard,Hernandez,XXXXXXXXX,XXXXXXXXX,6303 Heather Plaza,Brownsville,TX,78521,22945,2013-12-13 00:00:00.0,COMPLETE
2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126,15192,2013-10-29 00:00:00.0,PENDING_PAYMENT
2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126,33865,2014-02-18 00:00:00.0,COMPLETE
2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126,57963,2013-08-02 00:00:00.0,ON_HOLD
2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126,67863,2013-11-30 00:00:00.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...
12434,Mary,Mills,XXXXXXXXX,XXXXXXXXX,9720 Colonial Parade,Caguas,PR,725,42915,2014-04-16 00:00:00.0,COMPLETE
12434,Mary,Mills,XXXXXXXXX,XXXXXXXXX,9720 Colonial Parade,Caguas,PR,725,51800,2014-06-14 00:00:00.0,ON_HOLD
12434,Mary,Mills,XXXXXXXXX,XXXXXXXXX,9720 Colonial Parade,Caguas,PR,725,61777,2013-12-26 00:00:00.0,COMPLETE


In [49]:
customers_orders_df.shape # Display the shape of the resulting dataframe

(68883, 11)

In [50]:
customers_orders_df.reset_index()

,customer_id,customer_fname,customer_lname,customer_email,customer_password,customer_street,customer_city,customer_state,customer_zipcode,order_id,order_date,order_status
0,1,Richard,Hernandez,XXXXXXXXX,XXXXXXXXX,6303 Heather Plaza,Brownsville,TX,78521,22945,2013-12-13 00:00:00.0,COMPLETE
1,2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126,15192,2013-10-29 00:00:00.0,PENDING_PAYMENT
2,2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126,33865,2014-02-18 00:00:00.0,COMPLETE
3,2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126,57963,2013-08-02 00:00:00.0,ON_HOLD
4,2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126,67863,2013-11-30 00:00:00.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...
68878,12434,Mary,Mills,XXXXXXXXX,XXXXXXXXX,9720 Colonial Parade,Caguas,PR,725,42915,2014-04-16 00:00:00.0,COMPLETE
68879,12434,Mary,Mills,XXXXXXXXX,XXXXXXXXX,9720 Colonial Parade,Caguas,PR,725,51800,2014-06-14 00:00:00.0,ON_HOLD
68880,12434,Mary,Mills,XXXXXXXXX,XXXXXXXXX,9720 Colonial Parade,Caguas,PR,725,61777,2013-12-26 00:00:00.0,COMPLETE
68881,12435,Laura,Horton,XXXXXXXXX,XXXXXXXXX,5736 Honey Downs,Summerville,SC,29483,41643,2014-04-08 00:00:00.0,PENDING


In [ ]:
customers_orders_df.\
    groupby('customer_id').\
    agg({'order_id': 'count'}).\
    reset_index() # Count number of orders per customer

,customer_id,order_id
173,174,12
196,197,11
219,221,15
333,335,11
427,430,11
...,...,...
12191,12221,12
12196,12226,13
12254,12284,15
12299,12329,11


In [ ]:
customers_orders_df.\
    groupby('customer_id')['order_id'].\
    agg(no_of_orders='count').\
    reset_index() # Alternative way to count number of orders per customer

,customer_id,no_of_orders
0,1,1
1,2,4
2,3,7
3,4,6
4,5,4
...,...,...
12400,12431,16
12401,12432,10
12402,12433,4
12403,12434,8


In [56]:
customers_orders_df.\
    groupby('customer_id').\
    agg({'order_id': 'count'}).\
    reset_index().\
    query('order_id >= 10') # Filter customers with more than 10 orders

,customer_id,order_id
70,71,10
171,172,10
173,174,12
196,197,11
219,221,15
...,...,...
12311,12341,10
12317,12347,10
12375,12406,10
12400,12431,16


In [ ]:
customers_orders_df.\
    groupby('customer_id')['order_id'].\
    agg(no_of_orders='count').\
    reset_index().\
    query('no_of_orders >= 10') # Filter customers with more than 10 orders using alternative method    